# Buscador de jogos indie pelo que os jogadores dizem

**Problema:** a busca da Steam funciona por nome e tags. Este buscador encontra jogos pelo **conteúdo das reviews** dos jogadores brasileiros, com consultas como *"jogo relaxante sem combate"*.

**Corpus:** SteamBR (USP, 2023), pasta **Indie**. Cada documento é um jogo: o nome mais até 50 reviews.

| Bloco | O que faz |
|---|---|
| 1. Carregamento | Lê os JSONs e monta documentos e interações usuário-item |
| 2. Pré-processamento | 8 técnicas, uma função para cada |
| 3. Índice invertido | Dicionário de termos e postings `(documento, tf)` |
| 4. Modelos | Booleano, TF-IDF (cosseno) e BM25 |
| 5. Avaliação | Consultas, julgamento cego, P, R, F1 e nDCG |

## 0. Ambiente e configuração

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import glob, heapq, json, math, os, pickle, random, re, unicodedata
from collections import Counter, defaultdict
from functools import lru_cache

import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer

nltk.download('stopwords', quiet=True)   # lista de stopwords em português
nltk.download('rslp', quiet=True)        # stemmer para português
pd.set_option('display.max_colwidth', 50)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# Configurações de caminhos e infraestrutura do projeto
PASTA_CORPUS = '/content/drive/MyDrive/dados/Indie'
PASTA_SAIDA  = '/content/drive/MyDrive/SteamBR/saida'

# Otimização: extração local no disco do Colab para acelerar o processo de I/O
if not os.path.isdir(PASTA_CORPUS):
    !unzip -q -o "{ZIP_CORPUS}" -d /content/dados
os.makedirs(PASTA_SAIDA, exist_ok=True)
print(len(glob.glob(f'{PASTA_CORPUS}/*.json')), 'arquivos encontrados')

# Parâmetros globais estabelecidos para os limites do corpus e dos modelos
MAX_REVIEWS_POR_JOGO = 50
MIN_REVIEWS_POR_JOGO = 3
BM25_K1, BM25_B      = 1.2, 0.75
K                    = 10

3044 arquivos encontrados


## 1. Carregamento do corpus
Cada arquivo `Indie_<Nome>.json` traz as reviews de um jogo no formato da API da Steam.
- **Documento:** o nome do jogo mais as reviews mais úteis (`weighted_vote_score`), sem duplicatas.
- **Interação usuário-item:** cada review dá uma linha (usuário, jogo, recomendou?, tempo de jogo). É a base do Projeto 2.

In [6]:
def utilidade(review):
    # weighted_vote_score às vezes vem como texto ("0.52"), às vezes como 0
    try:
        return float(review.get('weighted_vote_score') or 0)
    except ValueError:
        return 0.0

def ler_jogo(caminho):
    with open(caminho, encoding='utf-8') as f:
        reviews = list(json.load(f).get('reviews', {}).values())
    doc_id = os.path.basename(caminho)[:-5]       # 'Indie_1bitHeart'
    nome = doc_id.split('_', 1)[-1]               # '1bitHeart'

    # Interações usuário-item: TODAS as reviews entram
    interacoes = [{'steamid': r['author']['steamid'], 'doc_id': doc_id,
                   'recomendou': r.get('voted_up'),
                   'minutos_jogados': r['author'].get('playtime_forever', 0)}
                  for r in reviews if r.get('author')]

    # Texto: reviews mais úteis primeiro, sem vazias e sem cópias idênticas
    textos, vistos = [], set()
    for r in sorted(reviews, key=lambda r: (utilidade(r), len(r.get('review') or '')), reverse=True):
        t = (r.get('review') or '').strip()
        if t and t.lower() not in vistos:
            vistos.add(t.lower())
            textos.append(t)
    textos = textos[:MAX_REVIEWS_POR_JOGO]

    documento = {'doc_id': doc_id, 'nome': nome, 'n_reviews': len(textos),
                 'texto': nome + '\n' + '\n'.join(textos)}   # o nome também é indexado
    return documento, interacoes

def carregar_corpus(pasta, fracao=1.0, semente=42):
    arquivos = sorted(glob.glob(f'{pasta}/*.json'))
    # Amostra aleatória com semente fixa: sempre os mesmos jogos e sem viés de ordem alfabética
    arquivos = random.Random(semente).sample(arquivos, int(len(arquivos) * fracao))

    documentos, interacoes = [], []
    for caminho in sorted(arquivos):
        try:
            doc, inter = ler_jogo(caminho)
        except (json.JSONDecodeError, UnicodeDecodeError) as erro:
            print('Arquivo ignorado:', caminho, erro)
            continue
        interacoes += inter
        if doc['n_reviews'] >= MIN_REVIEWS_POR_JOGO:
            documentos.append(doc)
    return documentos, interacoes

In [7]:
documentos, interacoes = carregar_corpus(PASTA_CORPUS)
print(f'Documentos (jogos): {len(documentos)}')

# Estrutura usuário-item
df_inter = pd.DataFrame(interacoes)
df_inter.to_csv(f'{PASTA_SAIDA}/interacoes_usuario_item.csv', index=False)
jogos_por_usuario = df_inter.groupby('steamid').size()
print(f'Interações: {len(df_inter)} | usuários: {len(jogos_por_usuario)} | jogos: {df_inter.doc_id.nunique()}')
print(f'Usuários que avaliaram 2+ jogos: {(jogos_por_usuario >= 2).sum()}')
print(f'Recomendações positivas: {df_inter.recomendou.mean():.1%}')

Documentos (jogos): 2975
Interações: 476285 | usuários: 284828 | jogos: 3044
Usuários que avaliaram 2+ jogos: 82883
Recomendações positivas: 93.6%


In [8]:
# Estrutura usuário-item
df_inter = pd.DataFrame(interacoes)
df_inter.to_csv(f'{PASTA_SAIDA}/interacoes_usuario_item.csv', index=False)
jogos_por_usuario = df_inter.groupby('steamid').size()
print(f'Interações: {len(df_inter)} | usuários: {len(jogos_por_usuario)} | jogos: {df_inter.doc_id.nunique()}')
print(f'Usuários que avaliaram 2+ jogos: {(jogos_por_usuario >= 2).sum()}')
print(f'Recomendações positivas: {df_inter.recomendou.mean():.1%}')

Interações: 476285 | usuários: 284828 | jogos: 3044
Usuários que avaliaram 2+ jogos: 82883
Recomendações positivas: 93.6%


## 2. Pré-processamento
Uma função por técnica, aplicada nesta ordem. **A mesma função processa documentos e consultas**; sem isso, os termos da consulta não casariam com os do índice.

| # | Técnica | Por quê (o que foi visto no corpus) |
|---|---|---|
| 1 | Limpeza de marcação | Reviews têm BBCode (`[b]`, `[spoiler]`) e checklists `( ) Multiplayer` / `(x) Offline`. A opção **não marcada** não descreve o jogo. |
| 2 | Case folding + remoção de acentos | Jogadores escrevem "nao" e "historia" sem acento. O acento sai **antes** do stemming, senão o RSLP gera radicais diferentes (`histor` ≠ `hist`). |
| 3 | Remoção de ruído | Risadas (`kkkk`, `hahaha`) e letras repetidas (`bastanteeee`). |
| 4 | Tokenização | Sequências de letras e números. Números ficam: "2d", "60 fps". |
| 5 | Expansão de abreviações | `n` é usado como "não" ("eu n enjoei"); também `mt`, `vc`, `pq`... |
| 6 | Marcação de negação | "não tem combate" → `nao_combate`. "Recomendo" e "não recomendo" precisam ser termos diferentes. |
| 7 | Stopwords | Lista NLTK-português **sem as negações**. A lista original remove "não", "nem" e "sem". |
| 8 | Stemming RSLP | Agrupa variações: jogo, jogos, jogar → `jog`. |

In [9]:
# ---- Recursos globais do pré-processamento ----
def sem_acento(texto):
    return ''.join(c for c in unicodedata.normalize('NFKD', texto) if not unicodedata.combining(c))

NEGACOES    = {'nao', 'nem', 'nunca', 'jamais', 'sem'}
AUXILIARES  = {'vou', 'vai', 'ter', 'la', 'tao', 'da'}       # Termos ignorados entre a negação e o verbo principal
ABREVIACOES = {'n': 'nao', 'naum': 'nao', 'mt': 'muito', 'mto': 'muito', 'vc': 'voce',
               'q': 'que', 'pq': 'porque', 'p': 'para', 'pra': 'para', 'tb': 'tambem',
               'tbm': 'tambem', 'msm': 'mesmo', 'vdd': 'verdade', 'eh': 'e', 'ta': 'esta'}
STOPWORDS_DOMINIO = set()                                     # Set de stopwords customizadas após análise do corpus
STOPWORDS = {sem_acento(p) for p in stopwords.words('portuguese')} - NEGACOES | STOPWORDS_DOMINIO
stemmer = RSLPStemmer()

# ---- TÉCNICA 1: limpeza de marcação ----
def limpar_marcacao(texto):
    texto = re.sub(r'https?://\S+', ' ', texto)                    # Remoção de URLs
    texto = re.sub(r'^\s*\(\s*\).*$', ' ', texto, flags=re.M)     # Filtro para ignorar opções não marcadas em checklists
    texto = re.sub(r'\(\s*[xX]\s*\)', ' ', texto)                  # Preservação do texto de opções selecionadas
    texto = re.sub(r'\[/?\w+[^\]]*\]|<[^<>\n]*>', ' ', texto)       # Remoção de tags BBCode e cabeçalhos HTML/XML
    return texto

# ---- TÉCNICA 2: case folding + remoção de acentos ----
def normalizar(texto):
    return sem_acento(texto.lower())

# ---- TÉCNICA 3: remoção de ruído informal ----
def remover_ruido(texto):
    texto = re.sub(r'\b\w*k{3,}\w*\b|\b[ks]{4,}\b', ' ', texto)  # Tratamento de risadas onomatopaicas (ex: kkkk)
    texto = re.sub(r'\b(?:ha){2,}h?\b|\b(?:rs){2,}\b', ' ', texto)
    texto = re.sub(r'(\w)\1{2,}', r'\1', texto)                    # Correção de alongamento vocálico (ex: bastanteeee -> bastante)
    return texto

# ---- TÉCNICA 4: tokenização ----
def tokenizar(texto):
    return re.findall(r'[a-z0-9]+', texto)

# ---- TÉCNICA 5: expansão de abreviações ----
def expandir_abreviacoes(tokens):
    return [ABREVIACOES.get(t, t) for t in tokens]

# ---- TÉCNICA 6: marcação de negação ----
def marcar_negacao(tokens):
    saida, negacao = [], None
    for t in tokens:
        if t in NEGACOES:
            negacao = t
        elif negacao and (t in STOPWORDS or t in AUXILIARES):
            continue
        elif negacao:
            saida.append(f'{negacao}_{t}')
            negacao = None
        else:
            saida.append(t)
    return saida

# ---- TÉCNICA 7: remoção de stopwords ----
def remover_stopwords(tokens):
    return [t for t in tokens if t not in STOPWORDS and (len(t) > 1 or t.isdigit())]

# ---- TÉCNICA 8: stemming RSLP ----
@lru_cache(maxsize=None)                             # Otimização de performance por cacheamento de radicais
def stem(t):
    if '_' in t:
        negacao, palavra = t.split('_', 1)
        return f'{negacao}_{stemmer.stem(palavra)}'
    return t if t.isdigit() else stemmer.stem(t)

# ---- PIPELINE COMPLETO ----
def preprocessar(texto):
    texto  = limpar_marcacao(texto)
    texto  = normalizar(texto)
    texto  = remover_ruido(texto)
    tokens = tokenizar(texto)
    tokens = expandir_abreviacoes(tokens)
    tokens = marcar_negacao(tokens)
    tokens = remover_stopwords(tokens)
    return [stem(t) for t in tokens]

In [10]:
# Validação prática do comportamento das etapas de transformação textual do pipeline
frase = 'Não vou mentir, eu n enjoei!! Jogo bassstanteeeee relaxante kkkkk, [b]sem combate[/b] e não é difícil.'
t = limpar_marcacao(frase);  print('1 limpeza     :', t)
t = normalizar(t);           print('2 normalização:', t)
t = remover_ruido(t);        print('3 ruído       :', t)
tk = tokenizar(t);           print('4 tokens      :', tk)	k = expandir_abreviacoes(tk); print('5 abreviações : ', tk)
tk = marcar_negacao(tk);     print('6 negação     :', tk)
tk = remover_stopwords(tk);  print('7 stopwords   :', tk)
print('8 stemming    :', [stem(x) for x in tk])

1 limpeza     : Não vou mentir, eu n enjoei!! Jogo bassstanteeeee relaxante kkkkk,  sem combate  e não é difícil.
2 normalização: nao vou mentir, eu n enjoei!! jogo bassstanteeeee relaxante kkkkk,  sem combate  e nao e dificil.
3 ruído       : nao vou mentir, eu n enjoei!! jogo bastante relaxante  ,  sem combate  e nao e dificil.
4 tokens      : ['nao', 'vou', 'mentir', 'eu', 'n', 'enjoei', 'jogo', 'bastante', 'relaxante', 'sem', 'combate', 'e', 'nao', 'e', 'dificil']
5 abreviações : ['nao', 'vou', 'mentir', 'eu', 'nao', 'enjoei', 'jogo', 'bastante', 'relaxante', 'sem', 'combate', 'e', 'nao', 'e', 'dificil']
6 negação     : ['nao_mentir', 'eu', 'nao_enjoei', 'jogo', 'bastante', 'relaxante', 'sem_combate', 'e', 'nao_dificil']
7 stopwords   : ['nao_mentir', 'nao_enjoei', 'jogo', 'bastante', 'relaxante', 'sem_combate', 'nao_dificil']
8 stemming    : ['nao_ment', 'nao_enjo', 'jog', 'bast', 'relax', 'sem_combat', 'nao_dificil']


### 2b. Análise quantitativa e qualitativa do corpus
Etapa de exploração estatística do vocabulário para embasar as decisões de filtragem e normalização:
- Identificação de negações e comportamento das stopwords padrão da NLTK.
- Frequência e abrangência dos termos de domínio para levantamento de potenciais ruídos.
- Avaliação empírica da taxa de redução léxica e possíveis casos de super-stemming do algoritmo RSLP.

In [11]:
nltk_original = {sem_acento(p) for p in stopwords.words('portuguese')}
freq, df_termos = Counter(), Counter()
for doc in documentos:
    tokens = expandir_abreviacoes(tokenizar(remover_ruido(normalizar(limpar_marcacao(doc['texto'])))))
    freq.update(tokens)
    df_termos.update(set(tokens) - nltk_original)

print('Negações que a lista NLTK original removeria:', {n: freq[n] for n in NEGACOES & nltk_original})
print('\nStopwords mais frequentes:', [(p, freq[p]) for p in sorted(nltk_original, key=lambda p: -freq[p])[:20]])
print('\nTermos de conteúdo presentes em mais jogos (candidatos a stopword de domínio):')
for termo, df in df_termos.most_common(20):
    print(f'  {termo:15s} {df / len(documentos):.0%} dos jogos')
print('\nPalavras em inglês:', [(p, freq[p]) for p in ['game', 'gameplay', 'the', 'good', 'soundtrack', 'coop']])

Negações que a lista NLTK original removeria: {'nem': 4020, 'sem': 7965, 'nao': 48563}

Stopwords mais frequentes: [('e', 193790), ('de', 116575), ('que', 99719), ('o', 98782), ('a', 83708), ('um', 64790), ('para', 56629), ('nao', 48563), ('com', 42297), ('muito', 39695), ('uma', 39577), ('voce', 36503), ('do', 36439), ('em', 34097), ('se', 32043), ('mas', 28921), ('mais', 27414), ('no', 24806), ('os', 24482), ('da', 22168)]

Termos de conteúdo presentes em mais jogos (candidatos a stopword de domínio):
  jogo            99% dos jogos
  bem             91% dos jogos
  bom             90% dos jogos
  game            82% dos jogos
  recomendo       81% dos jogos
  jogar           77% dos jogos
  tempo           76% dos jogos
  jogos           70% dos jogos
  vale            69% dos jogos
  simples         68% dos jogos
  divertido       68% dos jogos
  10              67% dos jogos
  ter             67% dos jogos
  pena            67% dos jogos
  vai             66% dos jogos
  boa      

In [12]:
# Análise léxica: mensuração da taxa de redução do vocabulário e agrupamentos de radicais
vocab = Counter()
for doc in documentos[:1000]:
    vocab.update(remover_stopwords(expandir_abreviacoes(tokenizar(remover_ruido(normalizar(limpar_marcacao(doc['texto'])))))))
grupos = defaultdict(list)
for palavra, n in vocab.most_common():
    grupos[stem(palavra)].append(palavra)
print(f'Vocabulário: {len(vocab)} palavras -> {len(grupos)} radicais (redução de {1 - len(grupos) / len(vocab):.0%})')
for radical, palavras in sorted(grupos.items(), key=lambda g: -len(g[1]))[:20]:
    print(f'  {radical:10s} <- {palavras[:8]}')

Vocabulário: 40835 palavras -> 21839 radicais (redução de 47%)
  jog        <- ['jogo', 'jogar', 'jogos', 'joguei', 'jogador', 'jogando', 'jogadores', 'joga']
  pass       <- ['passar', 'passa', 'passando', 'passado', 'passei', 'passou', 'passe', 'passo']
  fal        <- ['falar', 'falando', 'fala', 'falas', 'falei', 'falo', 'falam', 'falou']
  compr      <- ['comprar', 'comprei', 'compra', 'compre', 'comprem', 'comprado', 'compraria', 'comprando']
  melhor     <- ['melhor', 'melhores', 'melhorar', 'melhora', 'melhorou', 'melhorando', 'melhorada', 'melhorado']
  sent       <- ['sentido', 'sentir', 'senti', 'sente', 'sentimentos', 'sentimento', 'sentindo', 'sentidos']
  gost       <- ['gosta', 'gostei', 'gosto', 'gostar', 'gostaria', 'gostam', 'gostoso', 'gostou']
  lev        <- ['leve', 'levar', 'leva', 'levando', 'levei', 'leves', 'levam', 'levou']
  trabalh    <- ['trabalho', 'trabalhar', 'trabalhado', 'trabalhada', 'trabalhados', 'trabalha', 'trabalhando', 'trabalham']
  cont      

## 3. Índice invertido
- **Dicionário:** termo → postings.
- **Postings:** lista de `(documento, tf)` **em ordem crescente de documento**. A ordem permite fazer o AND do booleano por *merge*.
- **Guardado também:** o tamanho de cada documento (para o BM25) e a norma do vetor TF-IDF de cada documento (para o cosseno).
- **Sem posições:** não há busca por frase exata.

In [13]:
def idf(df, N):
    return math.log(N / df)

def construir_indice(documentos):
    postings = defaultdict(list)                     # termo -> [(doc, tf), ...]
    tamanhos = []
    for doc_num, doc in enumerate(documentos):       # doc_num cresce -> postings já saem ordenadas
        tokens = preprocessar(doc['texto'])
        tamanhos.append(len(tokens))
        for termo, tf in Counter(tokens).items():
            postings[termo].append((doc_num, tf))

    N = len(documentos)
    # Norma de cada documento no espaço TF-IDF (usada no cosseno)
    norma2 = [0.0] * N
    for termo, lista in postings.items():
        for doc_num, tf in lista:
            norma2[doc_num] += ((1 + math.log(tf)) * idf(len(lista), N)) ** 2

    return {'postings': dict(postings), 'N': N, 'tamanho': tamanhos,
            'tamanho_medio': sum(tamanhos) / N, 'norma': [math.sqrt(x) for x in norma2],
            'doc_ids': [d['doc_id'] for d in documentos], 'nomes': [d['nome'] for d in documentos]}

In [25]:
CAMINHO_INDICE = f'{PASTA_SAIDA}/indice.pkl'
if os.path.exists(CAMINHO_INDICE):
    indice = pickle.load(open(CAMINHO_INDICE, 'rb'))
else:
    indice = construir_indice(documentos)
    pickle.dump(indice, open(CAMINHO_INDICE, 'wb'))

postings, N = indice['postings'], indice['N']
print(f"Documentos: {N} | termos no dicionário: {len(postings)} | "
      f"postings: {sum(len(p) for p in postings.values())} | tamanho médio: {indice['tamanho_medio']:.0f} tokens")
termo = stem('relaxante')
print(f"Exemplo: '{termo}' aparece em {len(postings.get(termo, []))} jogos; primeiras postings: {postings.get(termo, [])[:5]}")

Documentos: 2975 | termos no dicionário: 44463 | postings: 1059043 | tamanho médio: 751 tokens
Exemplo: 'relax' aparece em 663 jogos; primeiras postings: [(8, 2), (14, 1), (18, 2), (19, 2), (32, 4)]


## 4. Modelos de recuperação
Os três usam **o mesmo índice e o mesmo pré-processamento**.

| Modelo | Consulta | Resultado |
|---|---|---|
| **Booleano** | `todos` (AND), `algum` (OR), `nenhum` (NOT) | Conjunto, sem ordem |
| **TF-IDF + cosseno** | Texto livre | Ranking por similaridade de vetores |
| **BM25** | Texto livre | Ranking probabilístico com saturação do tf e normalização por tamanho |

In [15]:
# ================= MODELO BOOLEANO =================
def intersecao(a, b):
    # Processamento em O(n + m) através do merge de ponteiros em listas ordenadas
    i = j = 0
    resultado = []
    while i < len(a) and j < len(b):
        if a[i] == b[j]:
            resultado.append(a[i]); i += 1; j += 1
        elif a[i] < b[j]:
            i += 1
        else:
            j += 1
    return resultado

def docs_com(termo):
    tokens = preprocessar(termo)
    if not tokens:
        return []
    resultado = [d for d, _ in postings.get(tokens[0], [])]
    for t in tokens[1:]:
        resultado = intersecao(resultado, [d for d, _ in postings.get(t, [])])
    return resultado

def buscar_booleano(consulta):
    resultado = list(range(N))
    for termo in consulta.get('todos', []):
        resultado = intersecao(resultado, docs_com(termo))
    if consulta.get('algum'):
        qualquer = set()
        for termo in consulta['algum']:
            qualquer |= set(docs_com(termo))
        resultado = intersecao(resultado, sorted(qualquer))
    proibidos = set()
    for termo in consulta.get('nenhum', []):
        proibidos |= set(docs_com(termo))
    return [d for d in resultado if d not in proibidos]

In [16]:
# ================= MODELO VETORIAL TF-IDF =================
# peso(t, d) = (1 + log tf) * log(N / df);   score = cosseno(consulta, documento)
def buscar_tfidf(texto, k=K):
    scores, norma_consulta = defaultdict(float), 0.0
    for termo, tf_q in Counter(preprocessar(texto)).items():
        lista = postings.get(termo)
        if not lista:
            continue
        peso_q = (1 + math.log(tf_q)) * idf(len(lista), N)
        norma_consulta += peso_q ** 2
        for doc, tf in lista:
            scores[doc] += peso_q * (1 + math.log(tf)) * idf(len(lista), N)
    if norma_consulta == 0:
        return []
    for doc in scores:                                  # produto escalar / (|d| * |q|)
        scores[doc] /= indice['norma'][doc] * math.sqrt(norma_consulta)
    return heapq.nlargest(k, scores.items(), key=lambda x: x[1])

In [17]:
# ================= MODELO PROBABILÍSTICO BM25 =================
def buscar_bm25(texto, k=K, k1=BM25_K1, b=BM25_B):
    scores = defaultdict(float)
    for termo in preprocessar(texto):
        lista = postings.get(termo)
        if not lista:
            continue
        df = len(lista)
        idf_bm25 = math.log(1 + (N - df + 0.5) / (df + 0.5))
        for doc, tf in lista:
            tamanho_relativo = indice['tamanho'][doc] / indice['tamanho_medio']
            # Aplicação dos parâmetros k1 (amortecimento de frequência) e b (normalização por tamanho)
            scores[doc] += idf_bm25 * tf * (k1 + 1) / (tf + k1 * (1 - b + b * tamanho_relativo))
    return heapq.nlargest(k, scores.items(), key=lambda x: x[1])

## 5. Avaliação experimental

### 5a. Consultas de teste e critérios de relevância
Os critérios de relevância e as consultas de teste foram integralmente especificados e documentados de forma estrita de modo a garantir a integridade metodológica e impedir qualquer tipo de viés posterior no processo de julgamento manual.

Cada consulta do teste contém:
- **texto:** string de busca livre avaliada pelos modelos estatísticos (TF-IDF e BM25).
- **booleana:** mapeamento estruturado de regras lógicas equivalentes.
- **criterio:** definição clara de relevância (0 = irrelevante, 1 = relevante, 2 = altamente relevante).

In [18]:
CONSULTAS = [
    {'id': 'q01', 'texto': 'jogo relaxante para desestressar',
     'booleana': {'algum': ['relaxante', 'tranquilo', 'calmo']},
     'criterio': 'Reviews descrevem o jogo como relaxante/calmo. 2 = principal atrativo; 1 = qualidade secundária.'},
    {'id': 'q02', 'texto': 'jogo relaxante sem combate',
     'booleana': {'algum': ['relaxante', 'tranquilo'], 'nenhum': ['combate']},
     'criterio': 'Jogo calmo E sem combate como mecânica principal.'},
    {'id': 'q03', 'texto': 'terror psicológico assustador',
     'booleana': {'todos': ['terror'], 'algum': ['psicologico', 'assustador', 'medo']},
     'criterio': 'Terror com foco em tensão psicológica. 1 = terror focado em sustos/ação.'},
    {'id': 'q04', 'texto': 'história emocionante que faz chorar',
     'booleana': {'todos': ['historia'], 'algum': ['chorar', 'emocionante', 'emociona']},
     'criterio': 'Narrativa com impacto emocional forte. 1 = história elogiada sem emoção forte.'},
    {'id': 'q05', 'texto': 'jogo cooperativo para jogar com amigos',
     'booleana': {'todos': ['amigos'], 'algum': ['coop', 'cooperativo']},
     'criterio': 'Modo cooperativo multijogador como ponto central.'},
    {'id': 'q06', 'texto': 'jogo muito difícil e desafiador',
     'booleana': {'todos': ['dificil'], 'algum': ['desafiador', 'desafio']},
     'criterio': 'Dificuldade alta é característica marcante.'},
    {'id': 'q07', 'texto': 'roguelike com runs viciantes',
     'booleana': {'algum': ['roguelike', 'roguelite']},
     'criterio': 'Jogo roguelike/roguelite (runs procedurais, morte permanente).'},
    {'id': 'q08', 'texto': 'metroidvania com exploração',
     'booleana': {'todos': ['metroidvania']},
     'criterio': 'Jogo no estilo metroidvania.'},
    {'id': 'q09', 'texto': 'pixel art bonita',
     'booleana': {'todos': ['pixel art'], 'algum': ['bonita', 'linda', 'lindo']},
     'criterio': 'Arte em pixel elogiada. 1 = pixel art presente mas não destacada.'},
    {'id': 'q10', 'texto': 'visual novel com escolhas e vários finais',
     'booleana': {'todos': ['finais'], 'algum': ['visual novel', 'escolhas']},
     'criterio': 'Jogo narrativo de escolhas com múltiplos finais.'},
    {'id': 'q11', 'texto': 'jogo sem bugs bem otimizado',
     'booleana': {'algum': ['sem bugs', 'otimizado']},
     'criterio': 'Reviews afirmam ausência de bugs ou boa otimização.'},
    {'id': 'q12', 'texto': 'jogo cheio de bugs que trava',
     'booleana': {'todos': ['bugs'], 'algum': ['trava', 'travando', 'crash']},
     'criterio': 'Bugs/travamentos relatados como problema relevante.'},
    {'id': 'q13', 'texto': 'roda em pc fraco',
     'booleana': {'algum': ['pc fraco', 'computador fraco', 'leve']},
     'criterio': 'Reviews afirmam que roda bem em computador modesto.'},
    {'id': 'q14', 'texto': 'jogo de plataforma 2d',
     'booleana': {'todos': ['plataforma', '2d']},
     'criterio': 'Jogo de plataforma em 2D.'},
    {'id': 'q15', 'texto': 'trilha sonora incrível',
     'booleana': {'algum': ['trilha sonora', 'soundtrack', 'ost']},
     'criterio': 'Música destacada como ponto forte. 1 = elogio breve.'},
    {'id': 'q16', 'texto': 'simulador de fazenda',
     'booleana': {'todos': ['fazenda'], 'algum': ['simulador', 'plantar', 'colheita']},
     'criterio': 'Simulação de fazenda/agricultura.'},
    {'id': 'q17', 'texto': 'jogo curto para terminar em poucas horas',
     'booleana': {'todos': ['curto', 'horas']},
     'criterio': 'Duração curta (até ~5h) como característica.'},
    {'id': 'q18', 'texto': 'puzzle inteligente que faz pensar',
     'booleana': {'algum': ['puzzle', 'puzzles', 'quebra cabeca']},
     'criterio': 'Quebra-cabeças de raciocínio lógico.'},
    {'id': 'q19', 'texto': 'não vale a pena muito caro',
     'booleana': {'todos': ['caro', 'nao vale']},
     'criterio': 'Preço criticado como alto para o conteúdo.'},
    {'id': 'q20', 'texto': 'jogo de cartas deckbuilder estilo slay the spire',
     'booleana': {'algum': ['deckbuilder', 'cartas deck']},
     'criterio': 'Jogo de construção de baralho (deckbuilding).'},
]
print(len(CONSULTAS), 'consultas')

20 consultas


### 5b. Os três modelos lado a lado sobre as mesmas consultas

In [19]:
# Geração da visualização comparativa dos resultados retornados pelos diferentes modelos
def lado_a_lado(c, k=K):
    nome = indice['nomes']
    colunas = {
        'TF-IDF': [f'{nome[d]} ({s:.2f})' for d, s in buscar_tfidf(c['texto'], k)],
        'BM25':   [f'{nome[d]} ({s:.2f})' for d, s in buscar_bm25(c['texto'], k)],
    }
    res_bool = buscar_booleano(c['booleana'])
    colunas[f'Booleano ({len(res_bool)} jogos, sem ordem)'] = [nome[d] for d in res_bool[:k]]
    n = max(len(v) for v in colunas.values())
    return pd.DataFrame({col: v + [''] * (n - len(v)) for col, v in colunas.items()}, index=range(1, n + 1))

# Amostra das 3 primeiras queries configuradas para visualização analítica inicial
for c in CONSULTAS[:3]:
    print(f"\n{c['id']}: {c['texto']}   |   booleana: {c['booleana']}")
    display(lado_a_lado(c))


q01: jogo relaxante para desestressar   |   booleana: {'algum': ['relaxante', 'tranquilo', 'calmo']}


,TF-IDF,BM25,"Booleano (1023 jogos, sem ordem)"
1,ProtoRaider (0.15),Borealis (8.45),AMorticiansTale
2,Weable (0.15),SudokuUniverse (8.35),ARavenMonologue
3,Borealis (0.12),ProtoRaider (8.31),AVICIIInvector
4,ColorinGame2 (0.12),PixelJunkEden (8.16),AlicesPatchwork
5,Asteria (0.12),ColorinGame2 (8.09),AlicesPatchworks2
6,SudokuUniverse (0.11),PerfectVermin (7.75),AlpacaStacka
7,PixelJunkEden (0.11),TheAdventureofNAYU (7.20),AlterWorld
8,PerfectVermin (0.10),LinesX (6.87),AmazingPrincessSarah
9,TheAdventureofNAYU (0.10),Pongo (6.82),AmericanRailroadsSUmmitRiver_PineValley
10,Wayward (0.10),ColoringGame4 (6.45),Anarcute



q02: jogo relaxante sem combate   |   booleana: {'algum': ['relaxante', 'tranquilo'], 'nenhum': ['combate']}


,TF-IDF,BM25,"Booleano (768 jogos, sem ordem)"
1,Sable (0.14),Sable (10.55),AMorticiansTale
2,Sundered (0.04),SKYE (3.14),ARavenMonologue
3,MyLifeChanged (0.03),Colors (3.11),AVICIIInvector
4,Nonogram (0.03),ColorinGame2 (3.11),AlicesPatchwork
5,NpcProblemsVertexColoring (0.03),ColoringGame23 (3.10),AlicesPatchworks2
6,InvisiblePlaces (0.03),CoLoringGame (3.08),AlpacaStacka
7,MinimalMaze (0.03),ColoringPIXELS (3.06),AlterWorld
8,PuzzlePetFeedyourcat (0.03),VirtualCottage (3.06),AmazingPrincessSarah
9,Scalak (0.03),Viridi (3.06),AmericanRailroadsSUmmitRiver_PineValley
10,MatchingCats (0.03),NinoMazeLOFI (3.05),Anarcute



q03: terror psicológico assustador   |   booleana: {'todos': ['terror'], 'algum': ['psicologico', 'assustador', 'medo']}


,TF-IDF,BM25,"Booleano (233 jogos, sem ordem)"
1,TheLandofPain (0.26),MissingChildren (13.68),ALongWayDown
2,ThePaperman (0.23),TheBoogieMan (13.46),ARobotNamedFight
3,MissingChildren (0.20),ThePaperman (13.43),Animality
4,DereEvilExe (0.19),IMSCARED (13.21),Ann
5,Horse (0.17),Horse (12.68),AnotherLostPhone
6,IronyOfNightmare (0.17),IronyOfNightmare (12.37),Anoxemia
7,LayersofFearInheritance (0.16),LayersofFearInheritance (12.18),Antenna
8,IMSCARED (0.16),ThroughtheWoods (12.15),AritanaandtheHarpysFeather
9,Morphine (0.15),Calendula (12.11),Atomicrops
10,FrightenedBeetles (0.14),Greetings (12.05),BaldisBasicsPlus


### 5c. Pool para julgamento cego
Para garantir a imparcialidade do processo de avaliação, foi gerado um pool unindo o top-10 do TF-IDF, o top-10 do BM25 e o conjunto recuperado pelo modelo booleano (limitado a até 30 jogos sorteados).

O arquivo resultante é embaralhado e desprovido de marcações sobre a origem do algoritmo. O arquivo final de julgamentos contendo as anotações manuais na coluna `relevancia` (com as notas de 0 a 2) é salvo como **`julgamentos.csv`** no diretório de saída.

In [26]:
CAMINHO_POOL = f'{PASTA_SAIDA}/pool_para_julgar.csv'
if os.path.exists(CAMINHO_POOL):
    print('Pool já existente:', CAMINHO_POOL)
else:
    random.seed(42)
    linhas = []
    for c in CONSULTAS:
        pool = {d for d, _ in buscar_tfidf(c['texto'])} | {d for d, _ in buscar_bm25(c['texto'])}
        res_bool = buscar_booleano(c['booleana'])
        pool |= set(random.sample(res_bool, min(30, len(res_bool))))
        pool = list(pool)
        random.shuffle(pool)                                   # Embaralhamento para ocultar origem dos algoritmos
        for d in pool:
            linhas.append({'qid': c['id'], 'consulta': c['texto'], 'criterio': c['criterio'],
                           'doc_id': indice['doc_ids'][d], 'jogo': indice['nomes'][d], 'relevancia': ''})
    pd.DataFrame(linhas).to_csv(CAMINHO_POOL, index=False)
    print(f'{len(linhas)} pares consulta-jogo prontos para julgamento em {CAMINHO_POOL}')

Pool já existente: /content/drive/MyDrive/SteamBR/saida/pool_para_julgar.csv


### 5d. Métricas
- **Relevante** = nota ≥ 1.
- **P@10, R@10, F1@10 e nDCG@10** para os rankings. O nDCG usa a nota graduada (0, 1, 2).
- **P, R e F1 sobre o conjunto inteiro** para o booleano, porque ele não tem ordem.
- **Recall de pool:** a revocação é calculada sobre os relevantes encontrados no pool, não sobre todos os relevantes da coleção.

In [24]:
# CSV com as reviews completas dos jogos do pool (para consultar enquanto julga)
docs_por_id = {d['doc_id']: d for d in documentos}
pool = pd.read_csv(CAMINHO_POOL)

linhas = []
for doc_id in pool['doc_id'].unique():
    d = docs_por_id[doc_id]
    # O texto do documento é composto pelo nome do jogo na primeira linha, seguido pelas reviews
    reviews = d['texto'].split('\n')[1:]
    for i, r_texto in enumerate(reviews, 1):
        linhas.append({'doc_id': doc_id, 'jogo': d['nome'], 'n': i,
                       'recomendou': 'N/D', 'review': r_texto})
pd.DataFrame(linhas).to_csv(f'{PASTA_SAIDA}/reviews_do_pool.csv', index=False)
print(f"{len(linhas)} reviews de {pool['doc_id'].nunique()} jogos -> reviews_do_pool.csv")

# Pool com trechos que mencionam termos da consulta (para julgar)
def trechos(doc_id, consulta, n=3, tamanho=250):
    termos = set(preprocessar(consulta))
    reviews = docs_por_id[doc_id]['texto'].split('\n')[1:]
    achados = [r for r in reviews if termos & set(preprocessar(r))]
    return ' || '.join(r[:tamanho].replace('\n', ' ') for r in (achados or reviews)[:n])

pool['trechos'] = [trechos(d, c) for d, c in zip(pool['doc_id'], pool['consulta'])]
pool.to_csv(f'{PASTA_SAIDA}/pool_com_contexto.csv', index=False)
print(len(pool), 'pares para julgar -> pool_com_contexto.csv')

103635 reviews de 715 jogos -> reviews_do_pool.csv
861 pares para julgar -> pool_com_contexto.csv


In [21]:
def precisao_revocacao_f1(recuperados, relevantes):
    acertos = len(set(recuperados) & relevantes)
    p = acertos / len(recuperados) if recuperados else 0.0
    r = acertos / len(relevantes) if relevantes else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f1

def ndcg(ranking, notas, k=K):
    dcg  = sum((2 ** notas.get(d, 0) - 1) / math.log2(i + 2) for i, d in enumerate(ranking[:k]))
    ideal = sorted(notas.values(), reverse=True)[:k]
    idcg = sum((2 ** g - 1) / math.log2(i + 2) for i, g in enumerate(ideal))
    return dcg / idcg if idcg else 0.0

def avaliar(julgamentos, b=BM25_B):
    resultados = []
    for c in CONSULTAS:
        notas = julgamentos.get(c['id'], {})            # doc_id -> nota
        if not notas:
            continue
        relevantes = {d for d, nota in notas.items() if nota >= 1}
        ids = indice['doc_ids']
        saidas = {'TF-IDF':   [ids[d] for d, _ in buscar_tfidf(c['texto'])],
                  'BM25':     [ids[d] for d, _ in buscar_bm25(c['texto'], b=b)],
                  'Booleano': [ids[d] for d in buscar_booleano(c['booleana'])]}
        for modelo, recuperados in saidas.items():
            p, r, f1 = precisao_revocacao_f1(recuperados, relevantes)
            resultados.append({'consulta': c['id'], 'modelo': modelo, 'P': p, 'R': r, 'F1': f1,
                               'nDCG@10': ndcg(recuperados, notas) if modelo != 'Booleano' else None})
    return pd.DataFrame(resultados)

In [27]:
jul = pd.read_csv(f'{PASTA_SAIDA}/julgamentos.csv').dropna(subset=['relevancia'])
julgamentos = defaultdict(dict)
for _, linha in jul.iterrows():
    julgamentos[linha['qid']][linha['doc_id']] = int(linha['relevancia'])

resultados = avaliar(julgamentos)
resultados.to_csv(f'{PASTA_SAIDA}/resultados.csv', index=False)
print('Média por modelo:')
display(resultados.groupby('modelo').mean(numeric_only=True).round(3))
print('F1 por consulta (onde cada modelo ganha ou perde):')
display(resultados.pivot(index='consulta', columns='modelo', values='F1').round(2))

Média por modelo:


,P,R,F1,nDCG@10
modelo,,,,
BM25,0.805,0.384,0.497,0.717
Booleano,0.129,0.955,0.189,NaN
TF-IDF,0.780,0.367,0.476,0.659


F1 por consulta (onde cada modelo ganha ou perde):


modelo,BM25,Booleano,TF-IDF
consulta,,,
q01,0.52,0.03,0.61
q02,0.61,0.06,0.48
q03,0.65,0.17,0.58
q04,0.43,0.17,0.43
q05,0.64,0.13,0.56
q06,0.46,0.05,0.51
q07,0.42,0.39,0.42
q08,0.43,0.60,0.43
q09,0.31,0.18,0.44


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


### 5e. Resumo Geral das Métricas Comparativas
Abaixo é consolidada a média geral de desempenho para os três modelos implementados sobre todo o conjunto de consultas avaliadas.

In [30]:
# Agrupamento e cálculo das médias gerais por modelo
resumo_metricas = resultados.groupby('modelo').mean(numeric_only=True).round(3)

# Reordenação e renomeação de colunas para melhor legibilidade no relatório
resumo_metricas = resumo_metricas.reindex(['Booleano', 'TF-IDF', 'BM25'])
resumo_metricas.columns = ['Precisão Média', 'Revocação Média', 'F1-Score Médio', 'nDCG@10 Médio']

display(resumo_metricas)

,Precisão Média,Revocação Média,F1-Score Médio,nDCG@10 Médio
modelo,,,,
Booleano,0.129,0.955,0.189,NaN
TF-IDF,0.780,0.367,0.476,0.659
BM25,0.805,0.384,0.497,0.717


In [29]:
# Experimento: efeito do parâmetro b do BM25 (normalização por tamanho do documento)
for b in [0.0, 0.25, 0.5, 0.75, 1.0]:
    r = avaliar(julgamentos, b=b)
    print(f"b={b}: ", r[r.modelo == 'BM25'][['P', 'R', 'F1', 'nDCG@10']].mean().round(3).to_dict())

b=0.0:  {'P': 0.325, 'R': 0.145, 'F1': 0.195, 'nDCG@10': 0.29}
b=0.25:  {'P': 0.465, 'R': 0.211, 'F1': 0.282, 'nDCG@10': 0.449}
b=0.5:  {'P': 0.63, 'R': 0.299, 'F1': 0.387, 'nDCG@10': 0.608}
b=0.75:  {'P': 0.805, 'R': 0.384, 'F1': 0.497, 'nDCG@10': 0.717}
b=1.0:  {'P': 0.7, 'R': 0.34, 'F1': 0.436, 'nDCG@10': 0.641}
